# Image-style LoRA for Network OS (Google Colab)

Trains `lora_style_training.py` on a free Colab GPU, because it is far too slow on a PC without an NVIDIA card.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Give it 10–20 images of the style you want, with a one-line caption each (cell 3).
3. It trains (about 20–30 minutes for 20 images × 100 epochs on a T4), shows images with and without the LoRA, and downloads `lora_output.zip`.
4. On your PC, unzip it into the project folder as `lora_output/` and load it with `pipe.load_lora_weights("lora_output")`.

What leaves your PC: only the images you upload, to your own Colab session. The base model is downloaded from Hugging Face inside Colab.

In [ ]:
# 1. Check the GPU is there. If this fails: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. The project code and the training libraries (Colab already has PyTorch).
!git clone --depth 1 https://github.com/DNA-Blockchain/Helloworld.git /content/network-os
%cd /content/network-os
!pip install -q diffusers peft transformers accelerate
# Colab's preinstalled torchao is too old for current peft, which then refuses to load; unused here.
!pip uninstall -q -y torchao


In [ ]:
# 3. Style images: upload a .zip of 10-20 images plus captions.json ({"img1.png": "caption", ...}),
#    all at the top level of the zip. maxwell_style_images.py can make them from Maxwell-chain blocks
#    you have locally; zip its output folder.
import json, os, zipfile
from google.colab import files
os.makedirs("style_images", exist_ok=True)
for name in files.upload():
    with zipfile.ZipFile(name) as z:
        z.extractall("style_images")

captions = json.load(open("style_images/captions.json"))
found = [f for f in captions if os.path.exists(os.path.join("style_images", f))]
print(f"{len(found)} images with captions; 10-20 is the useful range")
for f in found[:5]:
    print(" ", f, "->", captions[f])

In [ ]:
# 4. Train. Only the small LoRA weights learn; the base model stays frozen.
import lora_style_training as lst
setup = lst.build_training_pipeline(num_epochs=100, lora_rank=4, learning_rate=1e-4)
lst.train(setup)

In [ ]:
# 5. Same prompt and seed, without and with the LoRA.
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
PROMPT = captions[found[0]]
pipe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5", torch_dtype=torch.float16).to("cuda")
before = pipe(PROMPT, generator=torch.Generator("cuda").manual_seed(0), num_inference_steps=30).images[0]
pipe.load_lora_weights("lora_output")
after = pipe(PROMPT, generator=torch.Generator("cuda").manual_seed(0), num_inference_steps=30).images[0]
side_by_side = Image.new("RGB", (1024, 512))
side_by_side.paste(before, (0, 0)); side_by_side.paste(after, (512, 0))
print("left: base model   right: with the LoRA   prompt:", PROMPT)
side_by_side

In [ ]:
# 6. Download the LoRA weights (a few MB).
!cd /content/network-os && zip -qr lora_output.zip lora_output && ls -lh lora_output lora_output.zip
from google.colab import files
files.download("/content/network-os/lora_output.zip")